In [1]:
import networkx as nx;
import pandas as pd;
import gurobipy as gp;
from gurobipy import GRB;
import csv;
import sys;
import numpy

In [2]:
networkCSV = 'TestInstances/CSV_TestInstances/N25/' + 'N25_6.csv';
N = 100; #sample size
budget = 5;
numpy.random.seed(2024);

# Reading network file
with open(networkCSV, newline='') as f:
    reader = csv.reader(f);
    row1 = next(reader);
    nbArcs = int(row1[0]);
    row2 = next(reader);
    s = int(row2[0]);
    row3 = next(reader);
    t = int(row3[0]);
    
    G = nx.DiGraph();
    data = pd.read_csv(networkCSV, skiprows=4, header=None, delim_whitespace=True);
    n_edge = len(data.index);

    for i in range(n_edge): 
        G.add_edge(data.iat[i,0], data.iat[i,1], costLB = data.iat[i,2], 
                costUB = data.iat[i,3], interEffect = data.iat[i,4], tempCost = 0);

In [3]:
print("nbArcs = ", nbArcs);
print("s = ", s);
print("t = ", t);
print("G.nodes = ", G.nodes)
print("G.edges = ", G.edges)
for e in G.edges:
    print(e)
    print(G.edges[e])

nbArcs =  57
s =  1
t =  25
G.nodes =  [1, 7, 13, 14, 2, 19, 24, 3, 21, 4, 17, 5, 10, 6, 15, 11, 16, 8, 25, 9, 12, 18, 23, 20, 22]
G.edges =  [(1, 7), (1, 13), (1, 14), (7, 6), (7, 11), (7, 16), (7, 21), (13, 9), (14, 5), (14, 17), (14, 18), (2, 19), (2, 24), (19, 20), (19, 21), (19, 22), (24, 11), (3, 2), (3, 21), (21, 3), (21, 22), (4, 17), (4, 21), (17, 7), (5, 3), (5, 4), (5, 10), (5, 19), (5, 21), (10, 4), (10, 11), (10, 12), (10, 14), (10, 21), (6, 7), (6, 15), (15, 7), (15, 18), (11, 4), (11, 10), (11, 18), (16, 6), (16, 7), (16, 13), (16, 17), (16, 23), (8, 7), (8, 17), (8, 25), (9, 5), (12, 8), (12, 25), (18, 11), (23, 2), (20, 15), (22, 7), (22, 16)]
(1, 7)
{'costLB': 52.0, 'costUB': 66.0, 'interEffect': 7.0, 'tempCost': 0}
(1, 13)
{'costLB': 119.0, 'costUB': 119.0, 'interEffect': 7.0, 'tempCost': 0}
(1, 14)
{'costLB': 115.0, 'costUB': 135.0, 'interEffect': 6.0, 'tempCost': 0}
(7, 6)
{'costLB': 15.0, 'costUB': 15.0, 'interEffect': 7.0, 'tempCost': 0}
(7, 11)
{'costLB': 37.0, 

In [4]:
scens = [];
for k in range(N):
    scen = {};
    for e in G.edges:
        scen[e] = numpy.random.uniform(G.edges[e]['costLB'],G.edges[e]['costUB']);
    scens.append(scen);

In [6]:
# Callback - use lazy constraints
def lazy(model, where):
    if where == GRB.Callback.MIPSOL:
        xvals = model.cbGetSolution(model._x)
        thetavals = model.cbGetSolution(model._theta);
        for k in range(len(model._scens)):
            # multi-cut version
            # update edge cost per scenario
            for e in model._G.edges:
                if xvals[e] > 1e-5:
                    model._G.edges[e]['tempCost'] = model._scens[k][e] + model._G.edges[e]['interEffect'];
                else:
                    model._G.edges[e]['tempCost'] = model._scens[k][e];
            # obtain the shortest path and its length
            spValue = nx.shortest_path_length(model._G, source=model._s, target=model._t, weight='tempCost', method='dijkstra')
            if spValue < thetavals[k]*(1-(1e-5)):
                # add lazy constraints
                spPath = nx.shortest_path(model._G, source=model._s, target=model._t, weight='tempCost', method='dijkstra')
                constrCoefList = [1];
                constrVarList = [model._theta[k]];
                rhs = 0
                for i in range(len(spPath)-1):
                    rhs += model._scens[k][(spPath[i],spPath[i+1])];
                    constrCoefList.append(-model._G.edges[(spPath[i],spPath[i+1])]['interEffect']);
                    constrVarList.append(model._x[(spPath[i],spPath[i+1])]);
                expr = gp.LinExpr();
                expr.addTerms(constrCoefList, constrVarList);
                model.cbLazy(expr <= rhs);

In [7]:
master = gp.Model()

# Create variables
x = {};
for e in G.edges:
    x[e] = master.addVar(obj=0, vtype=GRB.BINARY);

theta = {};
for k in range(N):
    theta[k] = master.addVar(obj=1.0/N, vtype=GRB.CONTINUOUS, lb = 0, ub = 1e7);

# Add interdiction budget constraint
master.addConstr(gp.quicksum(x[e] for e in G.edges) <= budget);

master._x = x
master._theta = theta
master._G = G
master._s = s
master._t = t
master._scens = scens

Set parameter Username
Academic license - for non-commercial use only - expires 2024-07-30


In [8]:
master.modelSense = GRB.MAXIMIZE
master.Params.LazyConstraints = 1
master.optimize(lazy)

xvals = master.getAttr('X', x)

print('')
print('Optimal objval: %g' % master.ObjVal)
print('')
print('Optimal xval = ')
for e in G.edges:
    if xvals[e] > 1e-5:
        print(e);
        print(" ")

Set parameter LazyConstraints to value 1
Gurobi Optimizer version 9.5.0 build v9.5.0rc5 (mac64[x86])
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 1 rows, 67 columns and 57 nonzeros
Model fingerprint: 0x2e242af7
Variable types: 10 continuous, 57 integer (57 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e-01, 1e-01]
  Bounds range     [1e+00, 1e+07]
  RHS range        [5e+00, 5e+00]
Presolve time: 0.00s
Presolved: 1 rows, 67 columns, 57 nonzeros
Variable types: 10 continuous, 57 integer (57 binary)

Root relaxation: objective 3.321233e+02, 1 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

*    0     0               0     332.1233412  332.12334  0.00%     -    0s

Cutting planes:
  Lazy constraints: 10

Explored 1 nodes (1 simplex iterations) in 0.04 s